# Pipeline demonstration notebook



## Setup

### Setup segmentation model

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# print("Google Drive mounted successfully!")

Mounted at /content/drive
Google Drive mounted successfully!


In [ ]:
!pip install -q transformers datasets evaluate albumentations scikit-image opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


In [ ]:
import os
import os
import zipfile
import shutil
import random
import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm

import albumentations as A
import evaluate

from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation,
    get_cosine_schedule_with_warmup,
)


In [ ]:
import os

print("Searching for archive folder or archive.zip in Google Drive...")

possible_paths = [
    '/content/drive/MyDrive/archive',
    '/content/drive/MyDrive/archive.zip',
    '/content/drive/MyDrive/Colab Notebooks/archive',
    '/content/drive/MyDrive/Colab Notebooks/archive.zip',
    '/content/drive/My Drive/archive',
    '/content/drive/My Drive/archive.zip',
    '/content/drive/MyDrive/datasets/archive',
    '/content/drive/MyDrive/datasets/archive.zip',
]

archive_path = None
is_folder = False

for path in possible_paths:
    if os.path.exists(path):
        archive_path = path
        is_folder = os.path.isdir(path)
        print(f"✓ Found {'folder' if is_folder else 'zip file'} at: {path}")
        break

if not archive_path:
    print("✗ archive folder/zip not found in common locations.")
    print("\nSearching all archive files/folders in your Drive...")

    import subprocess
    result = subprocess.run(
        ['find', '/content/drive/MyDrive', '-name', 'archive', '-type', 'd'],
        capture_output=True,
        text=True,
        timeout=30
    )

    found_files = result.stdout.strip().split('\n')
    found_files = [f for f in found_files if f]

    if found_files:
        print(f"\nFound {len(found_files)} archive folder(s):")
        for i, f in enumerate(found_files, 1):
            print(f"  {i}. {f}")
        archive_path = found_files[0]
        is_folder = True
        print(f"\n✓ Using: {archive_path}")
    else:
        print("\n✗ No archive folder/zip found in Google Drive.")
        print("\nPlease manually set the path:")
        print("archive_path = '/content/drive/MyDrive/YOUR_PATH/archive'")
        print("is_folder = True")

Searching for archive folder or archive.zip in Google Drive...
✓ Found zip file at: /content/drive/MyDrive/archive.zip


In [ ]:
import zipfile
import os
import shutil

if 'archive_path' not in globals() or archive_path is None:
    raise FileNotFoundError("archive folder/zip not found. Please check the previous cell output.")

if is_folder:
    print(f"Using archive folder directly: {archive_path}")

    if archive_path != '/content/archive':
        print(f"Creating symlink to /content/archive...")
        if os.path.exists('/content/archive'):
            os.remove('/content/archive') if os.path.islink('/content/archive') else shutil.rmtree('/content/archive')
        os.symlink(archive_path, '/content/archive')
        print("✓ Symlink created")

    extract_dir = '/content/archive'
else:
    extract_dir = '/content/archive'
    os.makedirs(extract_dir, exist_ok=True)

    print(f"Extracting {archive_path}...")
    with zipfile.ZipFile(archive_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f"✓ Extracted to {extract_dir}")

train_files = os.listdir(os.path.join(extract_dir, 'train'))
val_files = os.listdir(os.path.join(extract_dir, 'valid'))
test_files = os.listdir(os.path.join(extract_dir, 'test'))

print(f"\nDataset structure:")
print(f"  Train: {len([f for f in train_files if f.endswith('_sat.jpg')])} images")
print(f"  Validation: {len([f for f in val_files if f.endswith('_sat.jpg')])} images")
print(f"  Test: {len([f for f in test_files if f.endswith('_sat.jpg')])} images")

Extracting /content/drive/MyDrive/archive.zip...
✓ Extracted to /content/archive

Dataset structure:
  Train: 803 images
  Validation: 171 images
  Test: 172 images


In [ ]:
# -----------------------------
# 2) Reproducibility
# -----------------------------
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

In [ ]:
# -----------------------------
# 3) Label mapping
# -----------------------------
id2label = {
    0: 'urban_land',
    1: 'agriculture_land',
    2: 'rangeland',
    3: 'forest_land',
    4: 'water',
    5: 'barren_land',
    6: 'unknown'
}
label2id = {v: k for k, v in id2label.items()}

COLOR_MAP = {
    (0, 255, 255): 0,
    (255, 255, 0): 1,
    (255, 0, 255): 2,
    (0, 255, 0): 3,
    (0, 0, 255): 4,
    (255, 255, 255): 5,
    (0, 0, 0): 6
}


In [ ]:
# -----------------------------
# 4) Dataset
# -----------------------------
class SemanticSegmentationDataset(Dataset):
    def __init__(self, root_dir, image_processor, train=True, image_size=512):
        self.root_dir = root_dir
        self.image_processor = image_processor
        self.train = train
        self.images = sorted([f for f in os.listdir(root_dir) if f.endswith('_sat.jpg')])
        assert len(self.images) > 0, f"No *_sat.jpg files found in {root_dir}"

        if train:
            self.transform = A.Compose([
                A.Resize(image_size, image_size),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.2),
                A.RandomRotate90(p=0.5),
                A.RandomBrightnessContrast(p=0.3),
                A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=10, val_shift_limit=10, p=0.2),
                A.ShiftScaleRotate(shift_limit=0.02, scale_limit=0.05, rotate_limit=10, border_mode=0, p=0.2),
                A.GaussNoise(var_limit=(5.0, 20.0), p=0.2),
            ])
        else:
            self.transform = A.Compose([
                A.Resize(image_size, image_size),
            ])

    def rgb_to_label(self, mask_rgb: np.ndarray) -> np.ndarray:
        label_mask = np.full(mask_rgb.shape[:2], fill_value=6, dtype=np.uint8)  # default unknown
        for color, class_idx in COLOR_MAP.items():
            matches = np.all(mask_rgb == np.array(color, dtype=np.uint8), axis=-1)
            label_mask[matches] = class_idx
        return label_mask

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        mask_name = img_name.replace('_sat.jpg', '_mask.png')

        img_path = os.path.join(self.root_dir, img_name)
        mask_path = os.path.join(self.root_dir, mask_name)

        image = np.array(Image.open(img_path).convert('RGB'))
        mask_rgb = np.array(Image.open(mask_path).convert('RGB'))
        mask = self.rgb_to_label(mask_rgb)

        transformed = self.transform(image=image, mask=mask)
        image = transformed['image']
        mask = transformed['mask']

        encoded = self.image_processor(
            images=image,
            segmentation_maps=mask,
            return_tensors="pt"
        )

        return {
            "pixel_values": encoded["pixel_values"].squeeze(0),
            "labels": encoded["labels"].squeeze(0).long(),
        }


In [ ]:
# -----------------------------
# 5) Processor / datasets / loaders
# -----------------------------
root_dir = '/dataset/train'
image_processor = SegformerImageProcessor(do_resize=False)

train_dataset = SemanticSegmentationDataset(os.path.join(root_dir, 'train'), image_processor, train=True, image_size=512)
valid_dataset = SemanticSegmentationDataset(os.path.join(root_dir, 'valid'), image_processor, train=False, image_size=512)
test_dataset  = SemanticSegmentationDataset(os.path.join(root_dir, 'test'),  image_processor, train=False, image_size=512)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"

num_workers = min(4, os.cpu_count() or 2)

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True,  num_workers=num_workers, pin_memory=pin_memory, persistent_workers=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=16, shuffle=False, num_workers=num_workers, pin_memory=pin_memory, persistent_workers=True)
test_dataloader  = DataLoader(test_dataset,  batch_size=16, shuffle=False, num_workers=num_workers, pin_memory=pin_memory, persistent_workers=True)

# sanity check
batch = next(iter(train_dataloader))
print({k: tuple(v.shape) for k, v in batch.items()})


In [ ]:
# -----------------------------
# 6) Model
# -----------------------------
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512",
    num_labels=7,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")


In [ ]:
# -----------------------------
# 7) Optimizer / scheduler
# -----------------------------
num_epochs = 100

optimizer = AdamW([
    {"params": model.segformer.parameters(), "lr": 1e-5},    # encoder — slow, preserve pretrained features
    {"params": model.decode_head.parameters(), "lr": 5e-4},  # head — fast, learning new task
], weight_decay=0.01)

num_training_steps = num_epochs * len(train_dataloader)
num_warmup_steps = max(1, int(0.1 * num_training_steps))
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

scaler = GradScaler("cuda", enabled=(device.type == "cuda"))
metric_name = "mean_iou"


In [ ]:
# -----------------------------
# 8) Train / eval loops
# -----------------------------
def train_epoch(model, dataloader, optimizer, scheduler, scaler, device, grad_accum_steps=2):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(enumerate(dataloader), total=len(dataloader), desc="Training")
    for step, batch in progress:
        pixel_values = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        with autocast(device_type="cuda", enabled=(device.type == "cuda")):
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss / grad_accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(dataloader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        running_loss += loss.item() * grad_accum_steps
        progress.set_postfix(loss=f"{running_loss / (step + 1):.4f}")

    return running_loss / len(dataloader)

@torch.no_grad()
def eval_epoch(model, dataloader, device):
    model.eval()
    metric = evaluate.load(metric_name)
    running_loss = 0.0

    progress = tqdm(dataloader, total=len(dataloader), desc="Evaluating")
    for batch in progress:
        pixel_values = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        with autocast(device_type="cuda", enabled=(device.type == "cuda")):
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss

        logits = outputs.logits
        upsampled_logits = torch.nn.functional.interpolate(
            logits,
            size=labels.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        preds = upsampled_logits.argmax(dim=1)

        metric.add_batch(
            predictions=preds.detach().cpu().numpy(),
            references=labels.detach().cpu().numpy(),
        )
        running_loss += loss.item()
        progress.set_postfix(loss=f"{running_loss / max(1, progress.n):.4f}")

    metrics = metric.compute(num_labels=7, ignore_index=255, reduce_labels=False)
    return running_loss / len(dataloader), metrics

def train_model(model, train_loader, valid_loader, optimizer, scheduler, scaler, device, num_epochs=10, patience=3):
    best_miou = -1.0
    patience_counter = 0
    best_path = "/content/drive/MyDrive/best_segformer_b0_7class.pth"

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, scaler, device, grad_accum_steps=2)
        val_loss, val_metrics = eval_epoch(model, valid_loader, device)

        mean_iou = float(val_metrics["mean_iou"])
        mean_accuracy = float(val_metrics["mean_accuracy"])

        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val Loss:   {val_loss:.4f}")
        print(f"Val mIoU:   {mean_iou:.4f}")
        print(f"Val mAcc:   {mean_accuracy:.4f}")

        if mean_iou > best_miou:
            best_miou = mean_iou
            patience_counter = 0
            torch.save({
                "model_state_dict": model.state_dict(),
                "id2label": id2label,
                "label2id": label2id,
            }, best_path)
            print(f"✓ Saved best model to {best_path}")
        else:
            patience_counter += 1
            print(f"No improvement. Patience {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("Early stopping.")
                break

    return model

In [ ]:
# -----------------------------
# 9) Start training
# -----------------------------
model = train_model(
    model,
    train_dataloader,
    valid_dataloader,
    optimizer,
    scheduler,
    scaler,
    device,
    num_epochs=num_epochs,
    patience=3,
)




Epoch 1/10


Training:   0%|          | 0/201 [00:00<?, ?it/s]

KeyboardInterrupt: 

### Setup synthesiser

In [ ]:
!pip install timm
!pip install rasterio

In [ ]:
import os
!pwd
os.chdir(os.path.join("drive","MyDrive","Colab Notebooks","swirsyntesiser"))


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from src.vit_model.mae import MaskedAutoencoderViT
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
model_ae = MaskedAutoencoderViT(
    img_size=224,
    patch_size=16,
    embed_dim=768,
    depth=12,
    num_heads=12,
    decoder_depth=2,
    mlp_ratio=4,
    decoder_num_heads=6,
    decoder_embed_dim=192,
)

In [ ]:

model_ae.load_state_dict(torch.load('latest.pth'))
model_ae = model_ae.cuda()
model_ae.eval()

## Segmentation

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

def ade_palette():
    return [
        [0, 255, 255],
        [255, 255, 0],
        [255, 0, 255],
        [0, 255, 0],
        [0, 0, 255],
        [255, 255, 255],
        [0, 0, 0]
    ]

image = valid_dataset[0]['original_image']

pixel_values = image_processor(image.permute(1, 2, 0).numpy(), return_tensors="pt").pixel_values.to("cuda")

model.eval()
with torch.no_grad():
    outputs = model(pixel_values=pixel_values)

predicted_segmentation_map = image_processor.post_process_semantic_segmentation(outputs, target_sizes=[[image.shape[1], image.shape[2]]])[0]
predicted_segmentation_map = predicted_segmentation_map.cpu()

color_seg = np.zeros((predicted_segmentation_map.shape[0], predicted_segmentation_map.shape[1], 3), dtype=np.uint8)

palette = np.array(ade_palette())
for label, color in enumerate(palette):
    color_seg[predicted_segmentation_map == label, :] = color

color_seg = color_seg[..., ::-1]

img = np.moveaxis(image.numpy(), 0, -1) * 0.5 + color_seg * 0.5

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(np.moveaxis(image.numpy(), 0, -1).astype(np.uint8))
plt.title('Original Image')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(img.astype(np.uint8))
plt.title('Prediction Overlay')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(valid_dataset[0]['original_map'].numpy())
plt.title('Ground Truth')
plt.axis('off')

plt.tight_layout()
plt.show()

print("=== Compression Section ===")
print("This section demonstrates ROI-based compression using your segmentation results.")
print("You can apply different compression strategies to ROI vs non-ROI regions.")

In [ ]:
img_mv = np.moveaxis(img,-1,0)
print(predicted_segmentation_map.shape)
resized_mask = F.interpolate(predicted_segmentation_map.unsqueeze(0).unsqueeze(0).to(torch.uint8), size = (224,224), mode='nearest').squeeze(0).squeeze(0)
print(resized_mask.shape)
resized_mask_np = resized_mask.numpy()





torch.Size([512, 512])
torch.Size([224, 224])


In [ ]:
from skimage.measure import block_reduce

print("Creating ROI blocks from segmentation mask...")

resized_mask = torch.nn.functional.interpolate(
    predicted_segmentation_map.unsqueeze(0).unsqueeze(0).float(),
    size=(224, 224),
    mode='nearest'
).squeeze(0).squeeze(0)

resized_mask_np = resized_mask.numpy()

block_size = (16, 16)
roi_block = block_reduce(resized_mask_np, block_size=block_size, func=np.mean)
roi_block = np.where(roi_block > 0.3, 1, 0)

roi_coords = []
for i in range(roi_block.shape[0]):
    for j in range(roi_block.shape[1]):
        if roi_block[i, j] == 1:
            x_start = i * block_size[0]
            y_start = j * block_size[1]
            x_end = x_start + block_size[0]
            y_end = y_start + block_size[1]
            roi_coords.append((x_start, y_start, x_end, y_end))

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(resized_mask_np, cmap='gray')
axes[0].set_title('Segmentation Mask (Pixel-level)')
axes[1].imshow(roi_block, cmap='gray', interpolation='nearest')
axes[1].set_title('Block-level ROI (16x16 blocks)')
plt.tight_layout()
plt.show()

print(f"Found {len(roi_coords)} ROI blocks")

In [ ]:
import cv2

def apply_bilateral_filter(image, mask, d=9, sigma_color=75, sigma_space=75):
    result = image.copy()
    if len(image.shape) == 3:
        for i in range(3):
            channel = image[:, :, i]
            filtered_channel = cv2.bilateralFilter(channel.astype(np.uint8), d, sigma_color, sigma_space)
            result[:, :, i] = np.where(mask[:, :, i], filtered_channel, channel)
    else:
        filtered = cv2.bilateralFilter(image.astype(np.uint8), d, sigma_color, sigma_space)
        result = np.where(mask, filtered, image)
    return result

print("Applying bilateral filter to non-ROI regions...")

original_img = valid_dataset[0]['original_image'].permute(1, 2, 0).numpy()

mask_2d = (resized_mask_np > 0).astype(bool)
mask_3d = np.repeat(mask_2d[:, :, np.newaxis], 3, axis=2)
inverse_mask_3d = ~mask_3d

resized_original = cv2.resize(original_img, (224, 224))

filtered_img = apply_bilateral_filter(resized_original, inverse_mask_3d, d=9, sigma_color=75, sigma_space=75)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(resized_original.astype(np.uint8))
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(filtered_img.astype(np.uint8))
axes[1].set_title('Bilateral Filtered (non-ROI)')
axes[1].axis('off')

diff = np.abs(resized_original.astype(float) - filtered_img.astype(float)).mean(axis=2)
axes[2].imshow(diff, cmap='hot')
axes[2].set_title('Difference Map')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("Bilateral filter applied to preserve edges while smoothing non-ROI regions")

In [ ]:
new_img = jpegxl_decode(compressed)

In [ ]:
print("Note: Synthesis/Extrapolation section requires Sentinel-2 .npz data format.")
print("Current dataset uses RGB .jpg images which don't have SWIR bands.")
print("This section is skipped for the archive dataset.")

In [ ]:
print("Note: NDMI/NDVI calculation requires NIR and SWIR bands from Sentinel-2 data.")
print("Current dataset uses RGB .jpg images which don't have these bands.")
print("This section is skipped for the archive dataset.")